In [2]:
import sqlite3
import pandas as pd
import numpy as np
conn = sqlite3.connect('laptops.db')

sql_query = """
    select L.Company, S.ScreenResolution, S.Inches, S.Cpu, S.Ram, S.Memory, S.Gpu, S.Weight, S.OpSys, S.Price
    from Laptops L
    join Specs S on L.laptop_id = S.laptop_id
"""
df = pd.read_sql_query(sql_query, conn)


In [18]:
df_org = pd.read_sql_query(sql_query, conn)

In [3]:
# ram
df['Ram'] = df['Ram'].str.replace('GB', '')
df['Ram'] = df['Ram'].astype(int)

In [4]:
# memory 
drive_type = ['SSD', 'HDD', 'Flash Storage', 'Hybrid']
def memory_extract(column, drive):
    extracted = column.str.extract(rf'(\d+)(GB|TB)\s+{drive}')
    size = pd.to_numeric(extracted[0])
    multiplier = extracted[1].map({'GB':1, 'TB':1024})
    df[f'{drive}_Size'] = (size * multiplier).fillna(0).astype(int)

for drive in drive_type:
    memory_extract(df['Memory'], drive)

In [5]:
#processing (weight)
df['Weight'] = df['Weight'].str.replace('kg', '')
df['Weight'] = df['Weight'].astype(float)

In [9]:
df.head(10)

,Company,ScreenResolution,Inches,Cpu,Ram,Memory,Gpu,Weight,OpSys,Price,SSD_Size,HDD_Size,Flash Storage_Size,Hybrid_Size,TouchScreen,IPS,PPI
0,Apple,IPS Panel Retina Display 2560x1600,13.3,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,1.37,macOS,71378.6832,128,0,0,0,0,1,226.983005
1,Apple,1440x900,13.3,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,1.34,macOS,47895.5232,0,0,128,0,0,0,127.677940
2,HP,Full HD 1920x1080,15.6,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,1.86,No OS,30636.0000,256,0,0,0,0,0,141.211998
3,Apple,IPS Panel Retina Display 2880x1800,15.4,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,1.83,macOS,135195.3360,512,0,0,0,0,1,220.534624
4,Apple,IPS Panel Retina Display 2560x1600,13.3,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,1.37,macOS,96095.8080,256,0,0,0,0,1,226.983005
5,Acer,1366x768,15.6,AMD A9-Series 9420 3GHz,4,500GB HDD,AMD Radeon R5,2.10,Windows 10,21312.0000,0,500,0,0,0,0,100.454670
6,Apple,IPS Panel Retina Display 2880x1800,15.4,Intel Core i7 2.2GHz,16,256GB Flash Storage,Intel Iris Pro Graphics,2.04,Mac OS X,114017.6016,0,0,256,0,0,1,220.534624
7,Apple,1440x900,13.3,Intel Core i5 1.8GHz,8,256GB Flash Storage,Intel HD Graphics 6000,1.34,macOS,61735.5360,0,0,256,0,0,0,127.677940
8,Asus,Full HD 1920x1080,14.0,Intel Core i7 8550U 1.8GHz,16,512GB SSD,Nvidia GeForce MX150,1.30,Windows 10,79653.6000,512,0,0,0,0,0,157.350512
9,Acer,IPS Panel Full HD 1920x1080,14.0,Intel Core i5 8250U 1.6GHz,8,256GB SSD,Intel UHD Graphics 620,1.60,Windows 10,41025.6000,256,0,0,0,0,1,157.350512


In [7]:
# screen resolution 
df['TouchScreen'] = df['ScreenResolution'].str.contains('Touchscreen', case=False).astype(int)
df['IPS'] = df['ScreenResolution'].str.contains('IPS', case=False).astype(int)

In [8]:
xy_resolution = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
x_res = xy_resolution[0].astype(int)
y_res = xy_resolution[1].astype(int)
df['PPI'] = np.sqrt((x_res)**2 + (y_res)**2) / df['Inches'].astype(float)

In [ ]:
# company
count = df['Company'].value_counts()
other = count[count <= 10].index
df['Company'] = df['Company'].replace(other, 'Other')